# CS383: Data Science and Machine Learning
## Lecture 8 Exercises — Logistic Regression & k-NN

Fill in every `__________` blank, then run all cells top to bottom. When you've completed this
notebook, download it (File → Save and Export Notebook As → Notebook (.ipynb), or the **Download**
button in the toolbar) and submit it on BrightSpace under **Lecture 8 Exercise** as a Jupyter Notebook
(.ipynb) file.

### Setup — NYC restaurant inspections

Run this first — same row-level dataset from the lecture: `is_critical` is the target, `score` /
`boro` / `cuisine_top` are the features.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score

try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score", "boro", "cuisine_description"]).reset_index(drop=True)
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)

    top_cuisines = inspections_df["cuisine_description"].value_counts().nlargest(10).index
    inspections_df["cuisine_top"] = np.where(
        inspections_df["cuisine_description"].isin(top_cuisines), inspections_df["cuisine_description"], "Other"
    )
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 4000
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza", "Japanese"]

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_top": rng.choice(cuisines_clean, size=n),
        "score": rng.integers(0, 71, size=n),
    })
    inspections_df["is_critical"] = rng.integers(0, 2, size=n)
    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(inspections_df):,} violation records")
inspections_df[["boro", "cuisine_top", "score", "is_critical"]].head()

---

### Step 1 — Split first, then check the baseline

Same order as always: split before you touch anything else. Then compute the majority-class baseline
accuracy on the *training* labels -- that's the number any real model has to beat.

In [ ]:
X = inspections_df[["score", "boro", "cuisine_top"]]
y = inspections_df["is_critical"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=__________, random_state=383, stratify=y)

majority_baseline = max(y_train.mean(), 1 - y_train.mean())
print("Training rows:", len(X_train))
print("Test rows:    ", len(X_test))
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

### Step 2 — Build a `ColumnTransformer` and fit logistic regression

Scale `score` and one-hot encode the two categorical columns -- k-NN in Step 4 will need the exact same
preprocessing, so build it carefully here.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", __________(), ["score"]),
    ("cat", __________(handle_unknown="ignore"), ["boro", "cuisine_top"]),
])

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

logreg_model = LogisticRegression(max_iter=1000)
logreg_model.__________(X_train_ready, y_train)

print("Training features shape:", X_train_ready.shape)

### Step 3 — Evaluate the logistic regression model

In [ ]:
y_pred_logreg = logreg_model.predict(X_test_ready)
y_proba_logreg = logreg_model.__________(X_test_ready)[:, 1]

logreg_accuracy = accuracy_score(y_test, y_pred_logreg)
logreg_auc = __________(y_test, y_proba_logreg)

print(f"Logistic regression accuracy: {logreg_accuracy:.3f}  (baseline: {majority_baseline:.3f})")
print(f"Logistic regression AUC:      {logreg_auc:.3f}")

### Step 4 — Fit k-NN on the same (already-scaled) features

Try a couple of different values of `k`.

In [ ]:
for k in [5, 15, 25]:
    knn_model = KNeighborsClassifier(n_neighbors=__________)
    knn_model.fit(X_train_ready, y_train)
    y_pred_knn = knn_model.predict(X_test_ready)
    knn_accuracy = accuracy_score(y_test, y_pred_knn)
    print(f"k={k:>2}: accuracy = {knn_accuracy:.3f}  (baseline: {majority_baseline:.3f}, logistic regression: {logreg_accuracy:.3f})")

### Step 5 — Explain it back

In 2-3 sentences: how did k-NN's accuracy compare to logistic regression's, across the different `k`
values you tried? For the smallest `k`, was k-NN's accuracy even above the majority-class baseline? What
does that tell you about using a more flexible model on a dataset with only modest, noisy signal?

**Your explanation:**

---

## Exercise 2 — Reflection (Exit Ticket)

Answer the following in your own words.

1. Why can't you just fit a `LinearRegression` on a 0/1 target and treat its output as a probability?
   Point to something specific that goes wrong.
2. In your own words, what does logistic regression's loss function (log-loss) punish more severely than
   a merely-slightly-off prediction?
3. Why doesn't logistic regression have a closed-form solution the way linear regression does, and what
   has to happen instead?
4. Sketch (in words) what would happen to k-NN's decision boundary if you set `k` equal to the entire
   size of the training set. Why?
5. What question do you still have about classification heading into Lecture 9 (Evaluation Metrics)?

**Your responses:**

1.
2.
3.
4.
5. 

## Optional Challenge

Apply the same workflow to a different real target: NYC 311's `resolution_time_hours`, turned into a
yes/no question -- was a complaint resolved **within 24 hours**?

### Setup — NYC 311

In [ ]:
import os

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_idx = rng.integers(0, n_days, size=n)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")
    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(complaint_types, size=n),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour
complaints_df = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)
complaints_df = complaints_df[complaints_df["resolution_time_hours"] >= 0].reset_index(drop=True)

# The target: did this complaint get resolved within 24 hours?
complaints_df["fast_resolution"] = (complaints_df["resolution_time_hours"] <= 24).astype(int)

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
print(f"fast_resolution balance: {complaints_df['fast_resolution'].mean():.3f}")

### Step 1 — Split, encode, and fit

Use `complaint_type`, `borough`, and `hour_filed` as features and `fast_resolution` as the target. Split
(don't forget `stratify=`), build a `ColumnTransformer` (scale `hour_filed`, one-hot the two
categoricals), then fit a `LogisticRegression`.

In [ ]:
# Your code here




### Step 2 — Evaluate

Compute accuracy, AUC, and the majority-class baseline. How does this compare to the restaurant dataset
in the main Lab?

In [ ]:
# Your code here




### Step 3 — Try k-NN here too

Fit a `KNeighborsClassifier` on the same preprocessed features, at a couple of different `k` values.

In [ ]:
# Your code here




### Step 4 — Reflect

Both this dataset and the restaurant dataset ask a real yes/no question about real NYC data -- but one
almost certainly classified far better than the other. Which one, and why do you think that is? (Hint:
think about what `complaint_type` alone might be telling you here that `score` alone couldn't tell you
about `is_critical`.)

**Your answer:**

### Big idea
> A classifier's accuracy is only ever as good as the signal actually sitting in your features, and the
> majority-class baseline is what tells you whether you found any at all. A fancier model -- more
> features, a smaller `k`, whatever -- is not automatically a better one; it can just as easily learn
> noise instead of pattern, especially when the real signal is thin. Always ask what you're predicting,
> from what, and whether the columns you handed the model ever had a fair shot at explaining it.